###Validação de Qualidade da Camada Ouro

Objetivo

Validar a qualidade técnica e a integridade da modelagem dimensional
construída na camada Ouro.

Serão avaliados:

- existência e volume das tabelas;
- unicidade das chaves das dimensões;
- granularidade da tabela fato;
- integridade referencial entre fato e dimensões;
- consistência dos domínios essenciais;
- completude das chaves obrigatórias;
- preservação dos registros provenientes da camada Prata.

Este notebook valida a modelagem da camada Ouro e não repete o
profiling detalhado realizado anteriormente na camada Prata.

In [0]:
# ============================================================
# Leitura e validação estrutural da camada Ouro
# ============================================================

from pyspark.sql import functions as F

TABELA_PRATA = "mvp3_eng_dados.prata.avaliacoes_harmonizadas"

TABELA_DIM_FONTE = "mvp3_eng_dados.ouro.dim_fonte"
TABELA_DIM_FILME = "mvp3_eng_dados.ouro.dim_filme"
TABELA_DIM_DATA = "mvp3_eng_dados.ouro.dim_data"
TABELA_FATO = "mvp3_eng_dados.ouro.fato_avaliacao"

df_prata = spark.table(TABELA_PRATA)

df_dim_fonte = spark.table(TABELA_DIM_FONTE)
df_dim_filme = spark.table(TABELA_DIM_FILME)
df_dim_data = spark.table(TABELA_DIM_DATA)
df_fato = spark.table(TABELA_FATO)

contratos = {
    "dim_fonte": (
        df_dim_fonte,
        {"id_fonte", "fonte"}
    ),
    "dim_filme": (
        df_dim_filme,
        {"id_filme", "titulo_filme", "link_filme"}
    ),
    "dim_data": (
        df_dim_data,
        {
            "id_data",
            "data_avaliacao",
            "ano",
            "trimestre",
            "mes",
            "dia"
        }
    ),
    "fato_avaliacao": (
        df_fato,
        {
            "id_avaliacao_global",
            "id_avaliacao",
            "id_fonte",
            "id_filme",
            "id_data",
            "tipo_avaliador",
            "texto_avaliacao",
            "classificacao_original",
            "sentimento",
            "nota_original",
            "data_avaliacao",
            "nome_avaliador",
            "publicacao",
            "eh_top_critic"
        }
    )
}

problemas_contrato = []

for nome, (df, colunas_esperadas) in contratos.items():

    ausentes = colunas_esperadas - set(df.columns)

    print(
        f"{nome}: "
        f"{len(df.columns)} colunas | "
        f"ausentes = {sorted(ausentes)}"
    )

    if ausentes:
        problemas_contrato.append(
            (nome, sorted(ausentes))
        )

assert len(problemas_contrato) == 0, (
    f"Problemas estruturais encontrados: "
    f"{problemas_contrato}"
)

print("\n✓ Contratos estruturais da Ouro validados.")

dim_fonte: 2 colunas | ausentes = []
dim_filme: 3 colunas | ausentes = []
dim_data: 6 colunas | ausentes = []
fato_avaliacao: 14 colunas | ausentes = []

✓ Contratos estruturais da Ouro validados.


In [0]:
# ============================================================
# Volumes e unicidade das chaves
# ============================================================

validacoes_chaves = [
    (
        "dim_fonte",
        df_dim_fonte.count(),
        df_dim_fonte.select("id_fonte").distinct().count()
    ),
    (
        "dim_filme",
        df_dim_filme.count(),
        df_dim_filme.select("id_filme").distinct().count()
    ),
    (
        "dim_data",
        df_dim_data.count(),
        df_dim_data.select("id_data").distinct().count()
    ),
    (
        "fato_avaliacao",
        df_fato.count(),
        df_fato.select("id_avaliacao_global").distinct().count()
    )
]

df_validacoes_chaves = spark.createDataFrame(
    validacoes_chaves,
    [
        "tabela",
        "registros",
        "chaves_distintas"
    ]
).withColumn(
    "duplicidades",
    F.col("registros") - F.col("chaves_distintas")
)

display(df_validacoes_chaves)

qtd_problemas_chave = (
    df_validacoes_chaves
    .filter(F.col("duplicidades") != 0)
    .count()
)

assert qtd_problemas_chave == 0, (
    "Foram encontradas chaves duplicadas na camada Ouro."
)

print("✓ Unicidade das chaves validada.")

tabela,registros,chaves_distintas,duplicidades
dim_fonte,2,2,0
dim_filme,17706,17706,0
dim_data,8015,8015,0
fato_avaliacao,1060128,1060128,0


✓ Unicidade das chaves validada.


In [0]:
# ============================================================
# Preservação da granularidade Prata → Ouro
# ============================================================

total_prata = df_prata.count()
total_ouro = df_fato.count()

ids_prata = (
    df_prata
    .select("id_avaliacao_global")
    .distinct()
    .count()
)

ids_ouro = (
    df_fato
    .select("id_avaliacao_global")
    .distinct()
    .count()
)

print(f"Registros Prata:       {total_prata:,}")
print(f"Registros Ouro:        {total_ouro:,}")
print(f"IDs distintos Prata:   {ids_prata:,}")
print(f"IDs distintos Ouro:    {ids_ouro:,}")

assert total_prata == total_ouro, (
    "A quantidade de registros foi alterada entre Prata e Ouro."
)

assert ids_prata == ids_ouro, (
    "A quantidade de avaliações distintas foi alterada."
)

print("\n✓ Granularidade preservada entre Prata e Ouro.")

Registros Prata:       1,060,128
Registros Ouro:        1,060,128
IDs distintos Prata:   1,060,128
IDs distintos Ouro:    1,060,128

✓ Granularidade preservada entre Prata e Ouro.


In [0]:
# ============================================================
# Integridade referencial da tabela fato
# ============================================================

orfãos_fonte = (
    df_fato
    .filter(F.col("id_fonte").isNotNull())
    .join(
        df_dim_fonte.select("id_fonte"),
        on="id_fonte",
        how="left_anti"
    )
    .count()
)

orfãos_filme = (
    df_fato
    .filter(F.col("id_filme").isNotNull())
    .join(
        df_dim_filme.select("id_filme"),
        on="id_filme",
        how="left_anti"
    )
    .count()
)

orfãos_data = (
    df_fato
    .filter(F.col("id_data").isNotNull())
    .join(
        df_dim_data.select("id_data"),
        on="id_data",
        how="left_anti"
    )
    .count()
)

print(f"Referências órfãs — fonte: {orfãos_fonte:,}")
print(f"Referências órfãs — filme: {orfãos_filme:,}")
print(f"Referências órfãs — data:  {orfãos_data:,}")

assert orfãos_fonte == 0
assert orfãos_filme == 0
assert orfãos_data == 0

print("\n✓ Integridade referencial validada.")

Referências órfãs — fonte: 0
Referências órfãs — filme: 0
Referências órfãs — data:  0

✓ Integridade referencial validada.


In [0]:
# ============================================================
# Completude das chaves e consistência do sentimento
# ============================================================

df_completude_fato = df_fato.agg(

    F.sum(
        F.when(
            F.col("id_avaliacao_global").isNull(),
            1
        ).otherwise(0)
    ).alias("sem_id_avaliacao_global"),

    F.sum(
        F.when(
            F.col("id_fonte").isNull(),
            1
        ).otherwise(0)
    ).alias("sem_id_fonte"),

    F.sum(
        F.when(
            F.col("id_filme").isNull(),
            1
        ).otherwise(0)
    ).alias("sem_id_filme"),

    F.sum(
        F.when(
            F.col("id_data").isNull(),
            1
        ).otherwise(0)
    ).alias("sem_id_data"),

    F.sum(
        F.when(
            F.col("sentimento").isNull()
            |
            (~F.col("sentimento").isin("POSITIVO", "NEGATIVO")),
            1
        ).otherwise(0)
    ).alias("sentimento_invalido")
)

display(df_completude_fato)

sem_id_avaliacao_global,sem_id_fonte,sem_id_filme,sem_id_data,sentimento_invalido
0,0,49712,49582,0


In [0]:
# ============================================================
# Validações críticas e cobertura dimensional
# ============================================================

metricas = df_completude_fato.first()

assert metricas["sem_id_avaliacao_global"] == 0, (
    "Existem avaliações sem identificador global."
)

assert metricas["sem_id_fonte"] == 0, (
    "Existem avaliações sem identificação de fonte."
)

assert metricas["sentimento_invalido"] == 0, (
    "Existem sentimentos nulos ou fora do domínio."
)

percentual_com_filme = (
    (total_ouro - metricas["sem_id_filme"])
    / total_ouro * 100
)

percentual_com_data = (
    (total_ouro - metricas["sem_id_data"])
    / total_ouro * 100
)

print("Validações críticas:")
print("✓ id_avaliacao_global completo")
print("✓ id_fonte completo")
print("✓ sentimento consistente")

print("\nCobertura dimensional:")

print(
    f"✓ avaliações com filme associado: "
    f"{percentual_com_filme:.2f}%"
)

print(
    f"✓ avaliações com data associada:  "
    f"{percentual_com_data:.2f}%"
)

Validações críticas:
✓ id_avaliacao_global completo
✓ id_fonte completo
✓ sentimento consistente

Cobertura dimensional:
✓ avaliações com filme associado: 95.31%
✓ avaliações com data associada:  95.32%


In [0]:
# ============================================================
# Resumo consolidado da qualidade
# ============================================================

print("=" * 68)
print("VALIDAÇÃO DE QUALIDADE — CAMADA OURO")
print("=" * 68)

print("\nVOLUMETRIA")
print(f"Registros Prata:                 {total_prata:,}")
print(f"Registros fato Ouro:             {total_ouro:,}")

print("\nUNICIDADE")
print(f"IDs de avaliação distintos:      {ids_ouro:,}")

print("\nINTEGRIDADE REFERENCIAL")
print(f"Referências órfãs de fonte:      {orfãos_fonte:,}")
print(f"Referências órfãs de filme:      {orfãos_filme:,}")
print(f"Referências órfãs de data:       {orfãos_data:,}")

print("\nCOMPLETUDE / CONSISTÊNCIA")
print(
    f"Sem id_avaliacao_global:         "
    f"{metricas['sem_id_avaliacao_global']:,}"
)
print(
    f"Sem id_fonte:                    "
    f"{metricas['sem_id_fonte']:,}"
)
print(
    f"Sem id_filme:                    "
    f"{metricas['sem_id_filme']:,}"
)
print(
    f"Sem id_data:                     "
    f"{metricas['sem_id_data']:,}"
)
print(
    f"Sentimentos inválidos:           "
    f"{metricas['sentimento_invalido']:,}"
)

print("\nCOBERTURA")
print(f"Com filme associado:             {percentual_com_filme:.2f}%")
print(f"Com data associada:              {percentual_com_data:.2f}%")

print("=" * 68)

VALIDAÇÃO DE QUALIDADE — CAMADA OURO

VOLUMETRIA
Registros Prata:                 1,060,128
Registros fato Ouro:             1,060,128

UNICIDADE
IDs de avaliação distintos:      1,060,128

INTEGRIDADE REFERENCIAL
Referências órfãs de fonte:      0
Referências órfãs de filme:      0
Referências órfãs de data:       0

COMPLETUDE / CONSISTÊNCIA
Sem id_avaliacao_global:         0
Sem id_fonte:                    0
Sem id_filme:                    49,712
Sem id_data:                     49,582
Sentimentos inválidos:           0

COBERTURA
Com filme associado:             95.31%
Com data associada:              95.32%


In [0]:
# ============================================================
# Encerramento
# ============================================================

print("=" * 68)
print("11_VALIDACAO_QUALIDADE_OURO — CONCLUÍDO")
print("=" * 68)

print("\nValidações concluídas:")
print("✓ contratos estruturais")
print("✓ unicidade das chaves")
print("✓ granularidade da tabela fato")
print("✓ preservação Prata → Ouro")
print("✓ integridade referencial")
print("✓ completude das chaves obrigatórias")
print("✓ consistência do domínio de sentimento")
print("✓ cobertura dimensional documentada")

print("\nCamada Ouro validada:")
print("✓ dim_fonte")
print("✓ dim_filme")
print("✓ dim_data")
print("✓ fato_avaliacao")

print(
    "\nPróxima etapa: "
    "12_analises_negocio"
)

11_VALIDACAO_QUALIDADE_OURO — CONCLUÍDO

Validações concluídas:
✓ contratos estruturais
✓ unicidade das chaves
✓ granularidade da tabela fato
✓ preservação Prata → Ouro
✓ integridade referencial
✓ completude das chaves obrigatórias
✓ consistência do domínio de sentimento
✓ cobertura dimensional documentada

Camada Ouro validada:
✓ dim_fonte
✓ dim_filme
✓ dim_data
✓ fato_avaliacao

Próxima etapa: 12_analises_negocio
